# Trabajo Final de Machine Learning - Ejercicio 3
## Clustering: perfiles de clientes mayoristas

**Grupo:** completar
**Integrantes:** completar

Se comparan K-Means y DBSCAN para identificar perfiles de clientes a partir de sus gastos anuales por categoria.

## 1. Metadata y planteamiento

El dataset Wholesale Customers contiene 440 clientes, con canal, region y gasto anual en Fresh, Milk, Grocery, Frozen, Detergents_Paper y Delicassen. Para clustering se excluyen `Channel` y `Region`, porque el objetivo es descubrir perfiles basados en comportamiento de compra, no reproducir etiquetas administrativas.

Pregunta: **es posible distinguir grupos de clientes con patrones de gasto diferentes?**

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

sns.set_theme(style='whitegrid', palette='deep')
RANDOM_STATE = 42

In [ ]:
DATA_PATH = Path('/Users/alexis/Downloads/Wholesale customers data.csv')
if not DATA_PATH.exists(): DATA_PATH = Path('Wholesale customers data.csv')
df = pd.read_csv(DATA_PATH)
print(f'Dimensiones: {df.shape}')
display(df.head())
display(df.describe().T)

In [ ]:
spending_cols = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
print('Valores faltantes:', df[spending_cols].isna().sum().sum())
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df[spending_cols], ax=axes[0], color='#56b4e9')
axes[0].set_title('Gasto por categoria (escala original)')
axes[0].tick_params(axis='x', rotation=45)
sns.heatmap(df[spending_cols].corr(), cmap='vlag', center=0, annot=True, fmt='.2f', ax=axes[1])
axes[1].set_title('Correlaciones de gasto')
plt.tight_layout()
plt.show()

# La transformacion log reduce el peso de valores extremos sin eliminar clientes.
X_log = np.log1p(df[spending_cols])
X_scaled = StandardScaler().fit_transform(X_log)

## 2. K-Means

Se usa `log1p` para reducir la asimetria de los gastos y luego se estandarizan las variables. Se comparan varios valores de K usando silhouette y Davies-Bouldin.

In [ ]:
k_rows = []
for k in range(2, 9):
    labels = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE).fit_predict(X_scaled)
    k_rows.append({'K': k, 'Silhouette': silhouette_score(X_scaled, labels), 'Davies-Bouldin': davies_bouldin_score(X_scaled, labels)})
k_results = pd.DataFrame(k_rows)
display(k_results.style.format({'Silhouette':'{:.3f}', 'Davies-Bouldin':'{:.3f}'}))
best_k = int(k_results.loc[k_results['Silhouette'].idxmax(), 'K'])
kmeans = KMeans(n_clusters=best_k, n_init=30, random_state=RANDOM_STATE)
df['kmeans_cluster'] = kmeans.fit_predict(X_scaled)
print('K elegido por silhouette:', best_k)

## 3. DBSCAN

DBSCAN identifica regiones densas y permite marcar observaciones atipicas como ruido (`-1`). Se prueba una pequeña grilla de hiperparametros y se descartan configuraciones con un unico grupo valido.

In [ ]:
db_rows = []
for eps in [0.7, 0.9, 1.1, 1.3, 1.5]:
    for min_samples in [4, 6, 10]:
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_scaled)
        valid = labels != -1
        groups = set(labels[valid])
        if valid.sum() > 1 and len(groups) >= 2:
            db_rows.append({'eps': eps, 'min_samples': min_samples, 'grupos': len(groups), 'ruido_%': 100 * (~valid).mean(), 'Silhouette_sin_ruido': silhouette_score(X_scaled[valid], labels[valid])})
db_results = pd.DataFrame(db_rows).sort_values('Silhouette_sin_ruido', ascending=False)
display(db_results.head(10).style.format({'ruido_%':'{:.1f}', 'Silhouette_sin_ruido':'{:.3f}'}))
if db_results.empty:
    raise ValueError('La grilla de DBSCAN no produjo al menos dos grupos validos.')
best_db = db_results.iloc[0]
dbscan = DBSCAN(eps=float(best_db['eps']), min_samples=int(best_db['min_samples']))
df['dbscan_cluster'] = dbscan.fit_predict(X_scaled)
print('DBSCAN seleccionado:', dict(best_db))

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X_scaled)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=df['kmeans_cluster'], palette='tab10', ax=axes[0], s=55)
axes[0].set_title('Perfiles K-Means')
sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=df['dbscan_cluster'], palette='tab10', ax=axes[1], s=55)
axes[1].set_title('Perfiles DBSCAN; -1 = ruido')
for ax in axes: ax.set_xlabel('Componente principal 1'); ax.set_ylabel('Componente principal 2')
plt.tight_layout()
plt.show()

profile = df.groupby('kmeans_cluster')[spending_cols].median().round(0)
display(profile)
print('Cantidad de clientes por cluster K-Means:')
display(df['kmeans_cluster'].value_counts().sort_index().rename('clientes').to_frame())
print('Cantidad de clientes por cluster DBSCAN:')
display(df['dbscan_cluster'].value_counts().sort_index().rename('clientes').to_frame())

## 4. Interpretacion y conclusiones

- K-Means entrega grupos compactos alrededor de centroides y permite resumir cada perfil con medianas de gasto.
- DBSCAN aporta una lectura complementaria: encuentra densidades y marca clientes atipicos sin obligarlos a pertenecer a un cluster.
- Los perfiles deben describirse comparando categorias relativas: por ejemplo, clientes orientados a Grocery/Detergents_Paper frente a clientes con mayor peso en Fresh/Frozen.
- Silhouette mas alto y Davies-Bouldin mas bajo indican mejor separacion relativa, pero no prueban que exista una segmentacion comercial definitiva.

**Conclusión:** el clustering permite explorar segmentos útiles para campañas y surtido, aunque conviene validar los perfiles con conocimiento del negocio, repetir el análisis en otros periodos y evaluar estabilidad ante cambios de escala o hiperparametros.